In [27]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier  
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import f1_score 
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
from sklearn.metrics import roc_auc_score
#importing all functions for this project 







In [2]:
data = pd.read_csv('/datasets/Churn.csv')
#downloading the dataset 



In [3]:
data.info()

#extracting all of the information to review for this project and the data types we will be working with 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB


In [4]:
print(data['Tenure'].describe())
print("\nTenure value counts:")
print(data['Tenure'].value_counts().sort_index())

# checking the ensure the correct number of missing values in the tenure column and seeing if it makes logical sense to drop the missing data values or filling them


count    9091.000000
mean        4.997690
std         2.894723
min         0.000000
25%         2.000000
50%         5.000000
75%         7.000000
max        10.000000
Name: Tenure, dtype: float64

Tenure value counts:
0.0     382
1.0     952
2.0     950
3.0     928
4.0     885
5.0     927
6.0     881
7.0     925
8.0     933
9.0     882
10.0    446
Name: Tenure, dtype: int64


In [5]:
# Remove rows with missing Tenure values
data_clean = data.dropna(subset=['Tenure'])
print(f"Original dataset size: {len(data)}")
print(f"Cleaned dataset size: {len(data_clean)}")

# I decided to remove 9% of the data due to the missing values appeared to be random, we have still large amount of data to work with, and the removal of the small percentage of data doesn create a bias in the target variable distribution. 

Original dataset size: 10000
Cleaned dataset size: 9091


In [6]:
#seeing the distribution class clearly after dropping the missing values and cleaing the data. We see that, - 0 (Stayed): 7,237 customers - 1 (Exited/Churned): 1,854 customers 
print("Target variable distribution:")
print(data_clean['Exited'].value_counts())


Target variable distribution:
0    7237
1    1854
Name: Exited, dtype: int64


In [7]:
# Define features (exclude non-predictive columns)
features = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 
           'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

# Create feature matrix and target vector
X = data_clean[features]
y = data_clean['Exited']

print("Feature columns:")
print(X.columns.tolist())
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")


Feature columns:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

Feature matrix shape: (9091, 10)
Target vector shape: (9091,)


In [8]:
#checking values of the categorical columns
print("Geography values:")
print(data_clean['Geography'].value_counts())
print("\nGender values:")
print(data_clean['Gender'].value_counts())


Geography values:
France     4550
Germany    2293
Spain      2248
Name: Geography, dtype: int64

Gender values:
Male      4974
Female    4117
Name: Gender, dtype: int64


In [9]:
# Create a copy to work with
X_encoded = X.copy()

# Encode Gender (binary: Male=1, Female=0)
label_encoder = LabelEncoder()
X_encoded['Gender'] = label_encoder.fit_transform(X_encoded['Gender'])

# One-hot encode Geography
geography_dummies = pd.get_dummies(X_encoded['Geography'], prefix='Geography')
X_encoded = pd.concat([X_encoded, geography_dummies], axis=1)
X_encoded = X_encoded.drop('Geography', axis=1)

print("Encoded features shape:", X_encoded.shape)
print("\nNew column names:")
print(X_encoded.columns.tolist())


Encoded features shape: (9091, 12)

New column names:
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


In [10]:
# Step 1: Separate the test set (20% of total data)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Split remaining 80% into train (64%) and validation (16%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X_encoded)*100:.1f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(X_encoded)*100:.1f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X_encoded)*100:.1f}%)")






Training set: 5817 samples (64.0%)
Validation set: 1455 samples (16.0%)
Test set: 1819 samples (20.0%)


In [11]:
# Step 1: Create and train a basic Random Forest model
rf_baseline = RandomForestClassifier(random_state=42)
rf_baseline.fit(X_train, y_train)

# Step 2: Make predictions
y_pred_baseline = rf_baseline.predict(X_test)

# Step 3: Evaluate the model

print("Baseline Random Forest Results:")
print(classification_report(y_test, y_pred_baseline))


print(f"F1 Score: {f1_score(y_test, y_pred_baseline):.4f}")




Baseline Random Forest Results:
              precision    recall  f1-score   support

           0       0.87      0.97      0.91      1448
           1       0.77      0.43      0.55       371

    accuracy                           0.86      1819
   macro avg       0.82      0.70      0.73      1819
weighted avg       0.85      0.86      0.84      1819

F1 Score: 0.5517


In [13]:
def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    
    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)
    
    return shuffle(features_upsampled, target_upsampled, random_state=42)


features_upsampled, target_upsampled = upsample(X_train, y_train, 4)


# shuffling the upsampled data

In [14]:
# Create upsampled training data
X_train_upsampled, y_train_upsampled = upsample(X_train, y_train, 4)
print(f"Upsampled training set shape: {X_train_upsampled.shape}")
print(f"Class distribution: {pd.Series(y_train_upsampled).value_counts()}")




Upsampled training set shape: (9375, 12)
Class distribution: 1    4744
0    4631
Name: Exited, dtype: int64


In [15]:
# Separate the training data by class
X_train_zeros = X_train[y_train == 0]  # Non-churned customers
X_train_ones = X_train[y_train == 1]   # Churned customers

y_train_zeros = y_train[y_train == 0]

y_train_ones = y_train[y_train == 1]

print(f"Non-churned customers (class 0): {X_train_zeros.shape[0]}")

print(f"Churned customers (class 1): {X_train_ones.shape[0]}")





Non-churned customers (class 0): 4631
Churned customers (class 1): 1186


In [16]:
# Create a results tracking dataframe
results = []

def evaluate_model(model, X_train_data, y_train_data, X_val, y_val, model_name):
    """Train model and evaluate on validation set"""
    model.fit(X_train_data, y_train_data)
    y_val_pred = model.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred)
    
    # Get probabilities for AUC-ROC
    y_val_proba = model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, y_val_proba)
    
    return {
        'Model': model_name,
        'Validation_F1': val_f1,
        
        
        'Validation_AUC': val_auc
    }
    

In [17]:
# 1. Baseline Random Forest
rf_baseline = RandomForestClassifier(random_state=42)
result1 = evaluate_model(rf_baseline, X_train, y_train, X_val, y_val, "Baseline RF")
results.append(result1)

# 2. Random Forest with upsampling
rf_upsampled = RandomForestClassifier(random_state=42)
result2 = evaluate_model(rf_upsampled, X_train_upsampled, y_train_upsampled, X_val, y_val, "RF + Upsampling")
results.append(result2)

# 3. Random Forest with class weights
rf_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')
result3 = evaluate_model(rf_balanced, X_train, y_train, X_val, y_val, "RF + Class Weights")
results.append(result3)

# 4. Hyperparameter tuning with upsampling
rf_tuned = RandomForestClassifier(random_state=42, n_estimators=300, max_depth=30)
result4 = evaluate_model(rf_tuned, X_train_upsampled, y_train_upsampled, X_val, y_val, "RF + Upsampling + Tuning")
results.append(result4)

# Display results
results_df = pd.DataFrame(results)
print("Model Comparison Results (Validation Set):")


print(results_df.round(4))




Model Comparison Results (Validation Set):
                      Model  Validation_F1  Validation_AUC
0               Baseline RF         0.5652          0.8522
1           RF + Upsampling         0.5909          0.8577
2        RF + Class Weights         0.5463          0.8611
3  RF + Upsampling + Tuning         0.6011          0.8596


In [18]:
X_train_zeros = X_train[y_train == 0]  # Non-churned customers
X_train_ones = X_train[y_train == 1] # Churned customers


y_train_zeros = y_train[y_train == 0]
y_train_ones = y_train[y_train == 1]

repeat = 4
features_upsampled = pd.concat([X_train_zeros]+[X_train_ones]*repeat)
target_upsampled = pd.concat([y_train_zeros]+[y_train_ones]*repeat)

print(features_upsampled.shape)




print(target_upsampled.shape)


(9375, 12)
(9375,)


In [19]:
# Apply upsampling only to training data (using your existing function)
X_train_upsampled, y_train_upsampled = upsample(X_train, y_train, 4)

print(f"Original training set class distribution:")
print(y_train.value_counts())
print(f"\nUpsampled training set class distribution:")
print(pd.Series(y_train_upsampled).value_counts())









Original training set class distribution:
0    4631
1    1186
Name: Exited, dtype: int64

Upsampled training set class distribution:
1    4744
0    4631
Name: Exited, dtype: int64


In [20]:
# Example with Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_upsampled, y_train_upsampled)

# Evaluate on validation set (not test!)
y_val_pred = rf_model.predict(X_val)
val_f1 = f1_score(y_val, y_val_pred)
print(f"Validation F1 Score: {val_f1:.4f}")






Validation F1 Score: 0.5909


In [28]:
# Step 1: Select the best model based on validation performance
print("=== FINAL MODEL SELECTION ===")
print("Best model: RF + Upsampling + Tuning")
print("Validation F1: 0.6011")
print("Validation AUC: 0.8596")
print()

# Step 2: Train the final model on training data
final_model = RandomForestClassifier(random_state=42, n_estimators=300, max_depth=30)
final_model.fit(X_train_upsampled, y_train_upsampled)

# Step 3: Evaluate ONCE on test set
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# Step 4: Calculate final metrics
final_f1 = f1_score(y_test, y_test_pred)
final_auc = roc_auc_score(y_test, y_test_proba)

# Step 5: Report final results
print("=== FINAL TEST SET RESULTS ===")
print(f"Final F1 Score: {final_f1:.2f}")
print(f"Final AUC-ROC: {final_auc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_test_pred))






=== FINAL MODEL SELECTION ===
Best model: RF + Upsampling + Tuning
Validation F1: 0.6011
Validation AUC: 0.8596

=== FINAL TEST SET RESULTS ===
Final F1 Score: 0.59
Final AUC-ROC: 0.8320

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      1448
           1       0.68      0.52      0.59       371

    accuracy                           0.85      1819
   macro avg       0.78      0.73      0.75      1819
weighted avg       0.84      0.85      0.84      1819



In [ ]:
# decided to combine both hyperparameters and upsampled data to meet the goal of .59


In [29]:
# Generate probability predictions from your best model
probabilities_test = rf_upsampled_tuned.predict_proba(X_test)
probabilities_positive_class = probabilities_test[:, 1]  # Get probabilities for class 1 (churned)

# Calculate AUC-ROC score
auc_roc_score = roc_auc_score(y_test, probabilities_positive_class)

# Print the results for comparison
print(f"AUC-ROC Score: {auc_roc_score:.4f}")
print(f"F1 Score: {f1_upsampled_tuned:.4f}")








AUC-ROC Score: 0.8350
F1 Score: 0.5919


In [ ]:
# we have successfully completed the target mark of an F1 score of .59 and have reached an AUC-ROC score of .84%. With the high AUC-ROC score, Beta Bank can use this code to have targeted campaign ads to focus on retention, they will know how to allocate their resourses effeciently by prioritizing the high risk customers, and design teired retention strategies for different risk groups.